<a href="https://colab.research.google.com/github/GarimaChaubey/NLP-Natural-Language-Preprocessing-/blob/main/Simple_RNN_Sentiment_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sentiment Classification Using a Simple RNN

**Beginner lab:** predict whether a short sentence is positive or negative.

Examples:
- `i love this movie` → Positive
- `i hate this movie` → Negative

We will use a tiny dataset so the code is easy to understand. The aim is to learn the RNN workflow, not to build a reliable real-world classifier.

## 1. Import libraries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import random

random.seed(1)
torch.manual_seed(1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

Using: cuda


## 2. Create a small dataset

`1` means positive and `0` means negative.

In [ ]:
sentences = [
    "i love this movie",
    "this movie is great",
    "i really like this",
    "what a wonderful movie",
    "this is a good film",
    "i enjoyed this movie",
    "i hate this movie",
    "this movie is terrible",
    "i really dislike this",
    "what a boring movie",
    "this is a bad film",
    "i hated this movie",
]

labels = [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0]

for sentence, label in zip(sentences, labels):
    print(sentence, "->", "Positive" if label == 1 else "Negative")

i love this movie -> Positive
this movie is great -> Positive
i really like this -> Positive
what a wonderful movie -> Positive
this is a good film -> Positive
i enjoyed this movie -> Positive
i hate this movie -> Negative
this movie is terrible -> Negative
i really dislike this -> Negative
what a boring movie -> Negative
this is a bad film -> Negative
i hated this movie -> Negative


## 3. Convert words into numbers

A neural network works with numbers, not raw words. We create a vocabulary and give each word an ID.

In [ ]:
vocabulary = {"<PAD>": 0}

for sentence in sentences:
    for word in sentence.lower().split():
        if word not in vocabulary:
            vocabulary[word] = len(vocabulary)

def sentence_to_ids(sentence):
    return [vocabulary[word] for word in sentence.lower().split() if word in vocabulary]

print("Vocabulary size:", len(vocabulary))
print("Sentence:", sentences[0])
print("Word IDs:", sentence_to_ids(sentences[0]))

Vocabulary size: 21
Sentence: i love this movie
Word IDs: [1, 2, 3, 4]


## 4. Make every sentence the same length

Padding adds zeros to shorter sentences so they can be stored in one tensor.

In [ ]:
MAX_LEN = max(len(s.split()) for s in sentences)

def encode_and_pad(sentence):
    ids = sentence_to_ids(sentence)[:MAX_LEN]
    ids += [0] * (MAX_LEN - len(ids))
    return ids

X = torch.tensor([encode_and_pad(s) for s in sentences], dtype=torch.long)
y = torch.tensor(labels, dtype=torch.long)

print("X shape (sentences, words):", X.shape)
print("y shape (labels):", y.shape)

X shape (sentences, words): torch.Size([12, 5])
y shape (labels): torch.Size([12])


## 5. Build the RNN model

The model has three parts:

1. **Embedding:** turns each word ID into a small vector.
2. **RNN:** reads the word vectors in order and updates its hidden state.
3. **Linear layer:** predicts two scores, one for positive and one for negative.

In [ ]:
class SentimentRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim=8, hidden_dim=16):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )
        self.classifier = nn.Linear(hidden_dim, 2)

    def forward(self, x):
        embedded = self.embedding(x)
        # embedded shape: [batch_size, sentence_length, embedding_dim]
        _, hidden = self.rnn(embedded)
        # hidden shape: [1, batch_size, hidden_dim]
        last_hidden = hidden[-1]
        return self.classifier(last_hidden)  # raw class scores (logits)

model = SentimentRNN(vocab_size=len(vocabulary)).to(device)
print(model)

SentimentRNN(
  (embedding): Embedding(21, 8, padding_idx=0)
  (rnn): RNN(8, 16, batch_first=True)
  (classifier): Linear(in_features=16, out_features=2, bias=True)
)


## 6. Train the model

- `CrossEntropyLoss` measures prediction error.
- `loss.backward()` calculates gradients.
- `optimizer.step()` updates the model's weights.

In [ ]:
X = X.to(device)
y = y.to(device)

loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

for epoch in range(1, 101):
    model.train()
    optimizer.zero_grad()

    logits = model(X)
    loss = loss_function(logits, y)

    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

Epoch 20, Loss: 0.0265
Epoch 40, Loss: 0.0009
Epoch 60, Loss: 0.0005
Epoch 80, Loss: 0.0003
Epoch 100, Loss: 0.0003


## 7. Predict sentiment

For a new sentence, we use the same vocabulary and padding rule. Words not seen during training are ignored in this tiny beginner example.

In [ ]:
def predict_sentiment(sentence):
    model.eval()
    ids = sentence_to_ids(sentence)
    ids = ids[:MAX_LEN] + [0] * max(0, MAX_LEN - len(ids))
    input_tensor = torch.tensor([ids], dtype=torch.long, device=device)

    with torch.no_grad():
        logits = model(input_tensor)
        predicted_label = int(logits.argmax(dim=1).item())

    return "Positive" if predicted_label == 1 else "Negative"

test_sentences = [
    "i love this movie",
    "this is a bad film",
    "i really like this",
    "this movie is terrible",
]

for sentence in test_sentences:
    print(f"{sentence} -> {predict_sentiment(sentence)}")

i love this movie -> Positive
this is a bad film -> Negative
i really like this -> Positive
this movie is terrible -> Negative


## 8. What did we build?

- The **embedding layer** represents each word as a vector.
- The **RNN** reads words in sequence and maintains a hidden state.
- The **final hidden state** is used as a sentence representation.
- The **classifier** predicts positive or negative sentiment.

### Try these small exercises
1. Add one new positive sentence and one new negative sentence, and update the labels.
2. Change `embedding_dim=8` to `16`.
3. Change `hidden_dim=16` to `32`.
4. Explain what the hidden state does.

**Limitation:** this dataset contains only 12 examples. It is useful for learning the code, but it cannot provide reliable predictions on general text.